### Reproyección de las coordenadas de WSG84 (las que tiene locBoyas) a EPSG32630 (las que usa el .tif)

In [8]:
import pandas as pd
from pyproj import Transformer

# Cargar el CSV
df = pd.read_csv("boyaUPCT/locBoyasUPCT.csv")

# Crear el transformador: de WGS84 (4326) a UTM zona 30N (EPSG:32630)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32630", always_xy=True)

# Aplicar transformación a cada punto
utm_x, utm_y = transformer.transform(df['Longitud'].values, df['Latitud'].values)

# Añadir columnas al DataFrame
df['LongitudEPSG32630'] = utm_x
df['LatitudEPSG32630'] = utm_y

# Guardar el resultado si quieres
df.to_csv("boyaUPCT/locBoyasUPCT_reproyectado.csv", index=False)

In [4]:
import os
import rasterio
import pandas as pd
import numpy as np

In [9]:
path = "boyaUPCT"
filename = "locBoyasUPCT_reproyectado.csv"
loc_boyas = pd.read_csv(os.path.join(path, filename)).iloc[:,:5]
loc_boyas.head(3)

,CodPuntoControl,Latitud,Longitud,LongitudEPSG32630,LatitudEPSG32630
0,CTD-1,37.811800,-0.784483,695024.711677,4.187246e+06
1,CTD-2,37.760617,-0.807800,693105.156141,4.181518e+06
2,CTD-3,37.761783,-0.783550,695238.475841,4.181698e+06


### .tif

In [13]:
folder_path = "Copernicus/SAFE_downloads/processed/"
filename = "S2A_MSIL1C_20180511T105031_N0500_R051_T30SXG_20230903T062608_C2XNets_10m.tif"
date = "2018-05-11"

tiff_file = os.path.join(folder_path, filename)  
results = []
with rasterio.open(tiff_file) as dataset:
    print(f"Processing {tiff_file}")
    bands = dataset.read()

    for _, row in loc_boyas.iterrows():
        buoy_id = row["CodPuntoControl"].replace('-', '').strip()
        lat = int(round(row["LatitudEPSG32630"]))
        lon = int(round(row["LongitudEPSG32630"]))

        try:
            row_idx, col_idx = dataset.index(lon, lat)
            #print(row_idx, col_idx)
            values = bands[:, row_idx, col_idx]
            results.append({
                "Date": date,
                "Buoy": buoy_id,
                "Latitude_EPSG32630": lat,
                "Longitude_EPSG32630": lon,
                "Latitutude_px": col_idx,
                "Longitude_px": row_idx,
                **{f"Band_{i+1}": val for i, val in enumerate(values)}
            })

        except IndexError:
            print(f"Skipping {buoy_id} on {date}: Coordinates out of raster bounds")
    results = pd.DataFrame(results)

Processing Copernicus/SAFE_downloads/processed/S2A_MSIL1C_20180511T105031_N0500_R051_T30SXG_20230903T062608_C2XNets_10m.tif


In [14]:
results

,Date,Buoy,Latitude_EPSG32630,Longitude_EPSG32630,Latitutude_px,Longitude_px,Band_1,Band_2,Band_3,Band_4,...,Band_7,Band_8,Band_9,Band_10,Band_11,Band_12,Band_13,Band_14,Band_15,Band_16
0,2018-05-11,CTD1,4187246,695025,729,109,0.017182,0.025626,0.039168,0.014868,...,0.003291,0.001286,0.016889,0.025546,0.039403,0.014756,0.011673,0.003324,0.363359,-0.000000e+00
1,2018-05-11,CTD2,4181518,693105,537,682,0.006423,0.016060,0.029806,0.006109,...,0.000922,0.000359,0.006337,0.016475,0.029672,0.006077,0.003948,0.000937,0.012153,-1.121039e-44
2,2018-05-11,CTD3,4181698,695238,750,664,0.022723,0.030132,0.034865,0.009466,...,0.001738,0.000683,0.022456,0.030076,0.035100,0.009365,0.006618,0.001749,0.158882,-0.000000e+00
3,2018-05-11,CTD4,4180266,698264,1053,807,0.007621,0.016598,0.024111,0.004739,...,0.000755,0.000299,0.007560,0.016859,0.024082,0.004692,0.003068,0.000752,0.006097,-0.000000e+00
4,2018-05-11,CTD5,4179450,700268,1253,889,0.005376,0.012977,0.015773,0.002098,...,0.000284,0.000111,0.005283,0.013423,0.015894,0.002042,0.001230,0.000286,0.000769,-2.869859e-42
5,2018-05-11,CTD6,4176009,695829,809,1233,0.009750,0.021306,0.035644,0.008684,...,0.001420,0.000553,0.009636,0.021542,0.035572,0.008692,0.005823,0.001417,0.058489,-0.000000e+00
6,2018-05-11,CTD7,4176724,690397,266,1161,0.022038,0.034885,0.049421,0.011335,...,0.001849,0.000709,0.022084,0.034728,0.049620,0.011377,0.007650,0.001866,0.202881,-0.000000e+00
7,2018-05-11,CTD8,4174178,693048,531,1416,0.008012,0.017644,0.026053,0.005021,...,0.000791,0.000313,0.007939,0.017937,0.026020,0.004972,0.003236,0.000788,0.006913,-0.000000e+00
8,2018-05-11,CTD9,4171106,693183,545,1723,0.012752,0.024302,0.046827,0.017611,...,0.003480,0.001338,0.012569,0.024385,0.046879,0.017550,0.013417,0.003571,0.292377,-0.000000e+00
9,2018-05-11,CTD10,4170388,695646,791,1795,0.025027,0.035097,0.048394,0.014305,...,0.002635,0.001020,0.024737,0.035047,0.048765,0.014211,0.010231,0.002651,0.277955,-0.000000e+00


### .dim

In [2]:
import sys
sys.path.append('/home/antonio/.snap/snap-python')
from esa_snappy import ProductIO, GeoPos

In [10]:
folder_path = "Copernicus/SAFE_downloads/export_dim/"
filename = "S2A_MSIL1C_20180511T105031_N0500_R051_T30SXG_20230903T062608_C2XNets_10m.dim"
date = "2018-05-11"


dim_file = os.path.join(folder_path, filename)
print(f"Usando archivo: {dim_file}")
product = ProductIO.readProduct(dim_file)

band_names = [b for b in product.getBandNames() if b.startswith("rhow")]
width = product.getSceneRasterWidth()
height = product.getSceneRasterHeight()

for _, row in loc_boyas.iterrows():
    buoy_id = str(row["CodPuntoControl"]).replace("-", "").strip()
    geo_coding = product.getSceneGeoCoding() # Obtiene sistema de georeferenciación del producto de snap

    lat = row["Latitud"]
    lon = row["Longitud"]

    # Crear objeto GeoPos y obtener posición de píxel
    geo_pos = GeoPos(lat, lon) # Objeto con ubicación geográfica (latitud, longitud en grados)
    pixel_pos = geo_coding.getPixelPos(geo_pos, None)  #Encontrar la posición de píxel en la imagen correspondiente a esas coordenadas geográficas

    # Redondear a índices de matriz
    x = int(round(pixel_pos.x))
    y = int(round(pixel_pos.y))

    try:
        reflectances = []
        for band_name in band_names:
            band = product.getBand(band_name)
            full_band = np.zeros(width * height, np.float32)
            band.readPixels(0, 0, width, height, full_band)
            full_band = full_band.reshape((height, width))
            value = full_band[y, x]
            reflectances.append(value)

        results.append({
            "Date": date,
            "Buoy": buoy_id,
            "Latitude_px": y,
            "Longitude_px": x,
            **{f"{band}": val for band, val in zip(band_names, reflectances)}
        })

    except Exception as e:
        print(f"Skipping {buoy_id} on {date}: {str(e)}")

product.dispose()
results = pd.DataFrame(results)

Usando archivo: Copernicus/SAFE_downloads/export_dim/S2A_MSIL1C_20180511T105031_N0500_R051_T30SXG_20230903T062608_C2XNets_10m.dim
Skipping CTD1 on 2018-05-11: name 'results' is not defined
Skipping CTD2 on 2018-05-11: name 'results' is not defined
Skipping CTD3 on 2018-05-11: name 'results' is not defined
Skipping CTD4 on 2018-05-11: name 'results' is not defined
Skipping CTD5 on 2018-05-11: name 'results' is not defined
Skipping CTD6 on 2018-05-11: name 'results' is not defined
Skipping CTD7 on 2018-05-11: index -418585148 is out of bounds for axis 0 with size 2255
Skipping CTD8 on 2018-05-11: name 'results' is not defined
Skipping CTD9 on 2018-05-11: name 'results' is not defined
Skipping CTD10 on 2018-05-11: name 'results' is not defined
Skipping CTD11 on 2018-05-11: name 'results' is not defined
Skipping CTD12 on 2018-05-11: name 'results' is not defined


NameError: name 'results' is not defined

In [ ]:
re